In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Florian_Wirtz_Problem

In [ ]:
# Base data from Kaggle only run on first train
import kagglehub
from data_loader import get_dataloader
import os
import yaml

# Download and locate data
path = kagglehub.dataset_download("hamzaboulahia/football-players-detection-dataset")
train_loader = get_dataloader(
    os.path.join(path, "players", "images", "train"),
    os.path.join(path, "players", "labels", "train")
)

data_config = {
    'path': path,
    'train': os.path.join(path, "players", "images", "train"),
    'val': os.path.join(path, "players", "images", "val"),
    'names': {
        0: 'player',
        1: 'referee',
        2: 'ball'
    }
}

# 3. Save the config file
with open('data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print(f"Data loaded: {len(train_loader)} batches ready.")

In [ ]:
# Custom data from Roboflow
!pip install roboflow --quiet

from roboflow import Roboflow
import os

# 2. Connect and Download
# Use the API key from your Roboflow settings (Settings > Workspace > API Keys)
rf = Roboflow(api_key=os.getenv("ROBOFLOW_KEY")
project = rf.workspace("keegans-workspace-a87iq").project("florian-wirtz-problem")
version = project.version("2")

# go into the data.yaml file located at florian-wirtz-problem and remove *_ from the labels

# This downloads the data into a local folder called 'datasets' in Colab
dataset = version.download("yolov11")

In [ ]:
%pip install ultralytics
from ultralytics import YOLO
import wandb
import shutil
import os

yolo = YOLO("runs/detect/wirtz_tracker/weights/best.pt")

# Train the model
yolo.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,         # Football needs about 100 to get the ball right
    imgsz=640,          # High enough to see the ball
    batch=16,           # Adjust based on your GPU memory
    device=0,           # Set to 0 for GPU, 'cpu' for CPU
    mosaic=1.0,         # Critical for sports: merges images to find small objects
    name="wirtz_tracker"
)

# 1. Initialize W&B
run = wandb.init(project="florian-wirtz-problem", job_type="export-artifact")

# 2. Define paths
best_model_path = 'runs/detect/wirtz_tracker/weights/best.pt'
export_folder = 'content/model_staging'

if os.path.exists(export_folder):
    shutil.rmtree(export_folder) # Clear the local staging area
os.makedirs(export_folder)

model = YOLO(best_model_path)
model.export(format='openvino', imgsz=640)

# 4. Prepare the Local Artifact
# Copy the PyTorch weights
shutil.copy(best_model_path, os.path.join(export_folder, 'soccer_model.pt'))

# Copy the OpenVINO folder
ov_source = best_model_path.replace('.pt', '_openvino_model')
if os.path.exists(ov_source):
    shutil.move(ov_source, os.path.join(export_folder, 'openvino_version'))

# 5. Log to W&B Artifacts (THIS handles the versioning)
artifact = wandb.Artifact(
    name='wirtz-tracking-model',
    type='model',
    description='YOLOv11s model trained on custom Roboflow soccer data'
)

# Add the entire folder to the artifact
artifact.add_dir(export_folder)

# Log it and give it a 'latest' alias automatically
run.log_artifact(artifact, aliases=["latest"])

print("complete")
run.finish()